# NorthForge Finance — End-to-End Workflow Run

Runs the full business workflow through `WorkflowOrchestrator`: Foundry's Trial Balance
pipeline (staging → enrichment → reporting → posting → interface) followed by the GL
import of that pipeline's Interface output — all under a single `WorkflowRun`. Recon is
then run explicitly against that same `workflow_run_id`, comparing Interface and GL.

Each section below reads and displays the data actually persisted at that stage, straight
from the domain repositories (`TrialBalanceRepository` for Foundry, `GLRepository` for
GL, `ReconClient` for recon).

## Spark session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/28 10:35:54 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/28 10:35:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-82c6d3b9-6237-4ccf-900e-9b8c9780f3c0;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 66ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

## Imports and helpers

In [2]:
from datetime import date

from pyspark.sql import functions as F

from core.logging import configure_logging
from foundry.pipeline import TrialBalancePipeline
from foundry.repository import TrialBalanceRepository
from foundry.config.settings import (
    CSV_TABLE_LOCATIONS,
    POSTGRES_TABLE_LOCATIONS
)

from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from spec import SpecClient
from atlas import AtlasClient
from reference import ReferenceClient

from registry import RegistryClient
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from workflow import WorkflowOrchestrator


configure_logging()


def display_df(df):
    display(df.toPandas())

## Configure clients and build the orchestrator

`TrialBalancePipeline` only knows Foundry processing now — no `RunTracker` —
`GLClient` only knows GL processing, and `ReconClient` only knows recon processing.
`WorkflowOrchestrator` is the thin layer that owns execution/workflow lifecycle across
Foundry and GL and coordinates Foundry → GL. Recon is not yet wired into the
orchestrator (v1 keeps it a standalone `recon.reconcile(workflow_run_id)` call,
run explicitly below once GL has posted).

In [3]:
BUSINESS_DT = date(2026, 3, 31)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

spec = SpecClient.from_db(
    spark,
    transformation_table = 'spec.transformation',
    file_layout_table = 'spec.file_layout'
    
)
atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)
reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

pipeline = TrialBalancePipeline(
    business_dt=BUSINESS_DT,
    repository=repository,
    spec=spec,
    atlas=atlas,
    reference=reference,
)

registry = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)
gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

orchestrator = WorkflowOrchestrator(
    run_tracker=run_tracker,
    foundry_pipeline=pipeline,
    gl=gl,
)

## Run the full workflow (Foundry → GL)

`run_workflow()` creates a single `WorkflowRun`, executes the Foundry pipeline
(`STAGING → ENRICHMENT → REPORTING → POSTING → INTERFACE`), then runs `GL / IMPORT`
against that same workflow's `FOUNDRY / INTERFACE` output — all as one business workflow.

In [4]:
workflow_result = orchestrator.run_workflow()

business_dt = pipeline.config.business_dt

# V1 has one producer execution per workflow per output table, so
# workflow_run_id alone is the operational key for every read below.
# gl_run_id (GL's own producer_run_id) is kept only as exact lineage.
workflow_run_id = workflow_result.foundry.identity.workflow_run_id
gl_run_id = workflow_result.gl.producer_run_id

2026-08-28 10:35:57,576 | INFO | workflow.orchestrator | Workflow started | workflow_run_id=764781eb-c9e0-4aa7-9eba-9736a81a662b | dataclass=TRIAL_BALANCE | business_dt=2026-03-31
2026-08-28 10:35:57,580 | INFO | workflow.orchestrator | Foundry pipeline started | run_id=d186a575-d392-4521-82a7-563a7f5149df | workflow_run_id=764781eb-c9e0-4aa7-9eba-9736a81a662b
2026-08-28 10:35:57,584 | INFO | workflow.orchestrator | Foundry zone started | operation=STAGING | run_id=5db12203-acb2-4a8f-9d41-6fba913248db | workflow_run_id=764781eb-c9e0-4aa7-9eba-9736a81a662b
2026-08-28 10:36:03,954 | INFO | workflow.orchestrator | Foundry zone succeeded | operation=STAGING | run_id=5db12203-acb2-4a8f-9d41-6fba913248db | workflow_run_id=764781eb-c9e0-4aa7-9eba-9736a81a662b | records=42
2026-08-28 10:36:03,959 | INFO | workflow.orchestrator | Foundry zone started | operation=ENRICHMENT | run_id=047d1559-1d2e-4f4c-b44f-e7f32229b747 | workflow_run_id=764781eb-c9e0-4aa7-9eba-9736a81a662b
2026-08-28 10:36:11,71

2026-08-28 10:36:19,997 | INFO | workflow.orchestrator | GL import succeeded | run_id=aab92ecd-e7ba-49b8-848b-54f7de826031 | workflow_run_id=764781eb-c9e0-4aa7-9eba-9736a81a662b | received=7 | posted=7 | rejected=0 | source_producer_run_id=ff1a0be0-d3ed-4258-8ff8-f569071224a4
2026-08-28 10:36:20,001 | INFO | workflow.orchestrator | Workflow succeeded | workflow_run_id=764781eb-c9e0-4aa7-9eba-9736a81a662b


## Foundry persistence layers

Each Foundry zone is read straight from its own persisted table, selected by the
`workflow_run_id` of the workflow that produced it (via `TrialBalanceRepository`) — not
by `business_dt`/`batch_id`. `PRODUCER_RUN_ID` remains stamped on every row as the exact
producer execution's lineage.

### Source

In [6]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,SRC_CLIENT_ID,CPTY_REF_ID,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD
0,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,80000.000000000000,USD
1,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,10000.000000000000,USD
2,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
3,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,90000.000000000000,USD
4,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_BACK_VALUED_ADJUSTMENT,USD,0E-12,USD
5,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_ADJUSTED_BALANCE,USD,90000.000000000000,USD
6,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,15000.000000000000,USD
7,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,5000.000000000000,USD
8,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
9,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,20000.000000000000,USD


### Staging

In [9]:
stg_df = repository.read_staging(workflow_run_id)

display_df(stg_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,POSTING_MEASURE_CCY_CD,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,10000.000000000000,1.000000000000,10000.000000000000,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,PREVIOUS_DAY_BALANCE,REPORTABLE,USD,80000.000000000000,1.000000000000,80000.000000000000,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,20000.000000000000,1.000000000000,20000.000000000000,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
7,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
8,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db
9,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,5000.000000000000,1.000000000000,5000.000000000000,DEBIT,764781eb-c9e0-4aa7-9eba-9736a81a662b,5db12203-acb2-4a8f-9d41-6fba913248db


### Enrichment

In [10]:
enr_df = repository.read_enrichment(workflow_run_id)

display_df(enr_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,047d1559-1d2e-4f4c-b44f-e7f32229b747
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,120000,220000,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,047d1559-1d2e-4f4c-b44f-e7f32229b747
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,USM,TRD,2000,LIABILITY,CREDIT,...,130000,210000,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,047d1559-1d2e-4f4c-b44f-e7f32229b747
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,USM,FIN,4000,REVENUE,CREDIT,...,410000,410000,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,047d1559-1d2e-4f4c-b44f-e7f32229b747
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,USM,FIN,3000,EQUITY,CREDIT,...,310000,310000,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,047d1559-1d2e-4f4c-b44f-e7f32229b747
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,CAM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,047d1559-1d2e-4f4c-b44f-e7f32229b747
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,CAM,TRD,2100,LIABILITY,CREDIT,...,140000,230000,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,047d1559-1d2e-4f4c-b44f-e7f32229b747


### Reporting

In [11]:
rpt_df = repository.read_reporting(workflow_run_id)

display_df(rpt_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,120000,220000,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
7,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
8,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c
9,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,764781eb-c9e0-4aa7-9eba-9736a81a662b,d88beb89-c467-41b6-a553-83404886e66c


### Posting

In [12]:
pst_df = repository.read_posting(workflow_run_id)

display_df(pst_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,POSTING_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,...,BACK_VALUE_ADJUSTED_BALANCE,ADJUSTED_BALANCE,POSTING_PREVIOUS_DAY_BALANCE,POSTING_CURRENT_DAY_DEBIT_BALANCE,POSTING_CURRENT_DAY_CREDIT_BALANCE,POSTING_CURRENT_DAY_EOD_BALANCE,POSTING_BACK_VALUE_ADJUSTED_BALANCE,POSTING_ADJUSTED_BALANCE,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,USM,TRD,1000,ASSET,...,0E-12,90000.000000000000,80000.000000000000,10000.000000000000,0E-12,90000.000000000000,0E-12,90000.000000000000,764781eb-c9e0-4aa7-9eba-9736a81a662b,81532e4a-ed33-48eb-a02b-5c2541e474dc
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,USM,FIN,1200,ASSET,...,0E-12,20000.000000000000,15000.000000000000,5000.000000000000,0E-12,20000.000000000000,0E-12,20000.000000000000,764781eb-c9e0-4aa7-9eba-9736a81a662b,81532e4a-ed33-48eb-a02b-5c2541e474dc
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,USM,TRD,2000,LIABILITY,...,0E-12,-65000.000000000000,-55000.000000000000,0E-12,-10000.000000000000,-65000.000000000000,0E-12,-65000.000000000000,764781eb-c9e0-4aa7-9eba-9736a81a662b,81532e4a-ed33-48eb-a02b-5c2541e474dc
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,USM,FIN,4000,REVENUE,...,0E-12,-25000.000000000000,-20000.000000000000,0E-12,-5000.000000000000,-25000.000000000000,0E-12,-25000.000000000000,764781eb-c9e0-4aa7-9eba-9736a81a662b,81532e4a-ed33-48eb-a02b-5c2541e474dc
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,USM,FIN,3000,EQUITY,...,0E-12,-20000.000000000000,-20000.000000000000,0E-12,0E-12,-20000.000000000000,0E-12,-20000.000000000000,764781eb-c9e0-4aa7-9eba-9736a81a662b,81532e4a-ed33-48eb-a02b-5c2541e474dc
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,CAM,TRD,1000,ASSET,...,1000.000000000000,51000.000000000000,40000.000000000000,10000.000000000000,0E-12,50000.000000000000,1000.000000000000,51000.000000000000,764781eb-c9e0-4aa7-9eba-9736a81a662b,81532e4a-ed33-48eb-a02b-5c2541e474dc
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,CAM,TRD,2100,LIABILITY,...,-1000.000000000000,-51000.000000000000,-40000.000000000000,0E-12,-10000.000000000000,-50000.000000000000,-1000.000000000000,-51000.000000000000,764781eb-c9e0-4aa7-9eba-9736a81a662b,81532e4a-ed33-48eb-a02b-5c2541e474dc


### Interface

This is the Foundry Interface output — the same rows GL reads as input for `GL / IMPORT`,
selected by `WORKFLOW_RUN_ID` rather than business date/batch.

In [13]:
int_df = repository.read_interface(workflow_run_id)

display_df(int_df)

,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,...,POSTING_STREAM,SRC_RECORD_ID,SRC_APP_CD,TRANSACTION_CURRENCY,TRANSACTION_AMOUNT,ACCOUNTED_CURRENCY,ACCOUNTED_AMOUNT,FX_RATE,AS_OF_DATE,BUSINESS_DATE
0,764781eb-c9e0-4aa7-9eba-9736a81a662b,ff1a0be0-d3ed-4258-8ff8-f569071224a4,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,1,1000,1100,NYC,101000,1000,...,GROSS_UP,rec-1,NFM,USD,90000.000000000000,USD,90000.000000000000,1.000000000000,2026-03-31,2026-03-31
1,764781eb-c9e0-4aa7-9eba-9736a81a662b,ff1a0be0-d3ed-4258-8ff8-f569071224a4,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,2,1000,1200,NYC,120000,1200,...,GROSS_UP,rec-2,NFM,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
2,764781eb-c9e0-4aa7-9eba-9736a81a662b,ff1a0be0-d3ed-4258-8ff8-f569071224a4,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,3,1000,1100,NYC,210000,2000,...,GROSS_UP,rec-3,NFM,USD,-65000.000000000000,USD,-65000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,764781eb-c9e0-4aa7-9eba-9736a81a662b,ff1a0be0-d3ed-4258-8ff8-f569071224a4,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,4,1000,1200,NYC,410000,4000,...,GROSS_UP,rec-4,NFM,USD,-25000.000000000000,USD,-25000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,764781eb-c9e0-4aa7-9eba-9736a81a662b,ff1a0be0-d3ed-4258-8ff8-f569071224a4,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,5,1000,1200,NYC,310000,3000,...,GROSS_UP,rec-5,NFM,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,764781eb-c9e0-4aa7-9eba-9736a81a662b,ff1a0be0-d3ed-4258-8ff8-f569071224a4,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-CAM-...,1,2000,2100,TOR,101000,1000,...,GROSS_UP,rec-6,NFM,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
6,764781eb-c9e0-4aa7-9eba-9736a81a662b,ff1a0be0-d3ed-4258-8ff8-f569071224a4,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-CAM-...,2,2000,2100,TOR,230000,2100,...,GROSS_UP,rec-7,NFM,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31


## GL persistence layer

`GL / IMPORT` (`gl_run_id`) reads the Interface rows produced by `FOUNDRY / INTERFACE`
above — selected by `workflow_run_id`, since V1 has one Interface producer execution per
workflow — and stamps its own execution identity onto whatever it writes: `gl.posting`
and `gl.rejection` rows carry `PRODUCER_RUN_ID = gl_run_id`, not the Interface producer's
ID. The reads below select by `workflow_run_id` too, not `gl_run_id`.

### GL Posting

In [14]:
gl_postings = gl.get_postings(workflow_run_id)

display_df(gl_postings)

,GL_POSTING_ID,POSTED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,FOUNDRY_RULE_ID,POSTING_ID,POSTING_STREAM,...,BOOK_CD,SOURCE_CD,CR_DR_IND,TRANSACTION_CURRENCY,TRANSACTION_AMOUNT,ACCOUNTED_CURRENCY,ACCOUNTED_AMOUNT,FX_RATE,AS_OF_DATE,BUSINESS_DATE
0,2b18d4c9-9f27-4763-9799-668c6e3e7b96,2026-08-28 10:36:15.093476,764781eb-c9e0-4aa7-9eba-9736a81a662b,aab92ecd-e7ba-49b8-848b-54f7de826031,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-CAM-...,1,TB-GROSS-UP,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
1,d78da423-bb12-460e-977c-776c4bde30fc,2026-08-28 10:36:16.636336,764781eb-c9e0-4aa7-9eba-9736a81a662b,aab92ecd-e7ba-49b8-848b-54f7de826031,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-CAM-...,2,TB-GROSS-UP,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31
2,63ebe6c0-dc44-46cc-96c3-afd3c0551031,2026-08-28 10:36:17.255956,764781eb-c9e0-4aa7-9eba-9736a81a662b,aab92ecd-e7ba-49b8-848b-54f7de826031,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,1,TB-GROSS-UP,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,USD,90000.000000000000,USD,90000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,dbe31336-795d-4239-8f22-97bfa9f82e3f,2026-08-28 10:36:17.898574,764781eb-c9e0-4aa7-9eba-9736a81a662b,aab92ecd-e7ba-49b8-848b-54f7de826031,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,2,TB-GROSS-UP,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,ddddfb91-948a-4e2d-819b-85cd73a21494,2026-08-28 10:36:18.539139,764781eb-c9e0-4aa7-9eba-9736a81a662b,aab92ecd-e7ba-49b8-848b-54f7de826031,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,3,TB-GROSS-UP,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-65000.000000000000,USD,-65000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,a6192191-df16-489d-b81a-c6cf321c6ab6,2026-08-28 10:36:19.184271,764781eb-c9e0-4aa7-9eba-9736a81a662b,aab92ecd-e7ba-49b8-848b-54f7de826031,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,4,TB-GROSS-UP,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-25000.000000000000,USD,-25000.000000000000,1.000000000000,2026-03-31,2026-03-31
6,ae35879e-56fe-414e-a4a1-1c0073d33fa3,2026-08-28 10:36:19.861048,764781eb-c9e0-4aa7-9eba-9736a81a662b,aab92ecd-e7ba-49b8-848b-54f7de826031,TRIAL_BALANCE,NFTB-047d1559-1d2e-4f4c-b44f-e7f32229b747-USM-...,5,TB-GROSS-UP,PST-260331-260331-81532e4a-ed33-48eb-a02b-5c25...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31


### GL Rejection

A non-zero rejected count here is a normal business outcome, not an execution failure —
`GL / IMPORT` still completes as `SUCCEEDED` as long as processing itself ran cleanly.

In [15]:
gl_rejections = gl.get_rejections(workflow_run_id)

display_df(gl_rejections)

,GL_REJECTION_ID,REJECTED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,FOUNDRY_RULE_ID,POSTING_ID,POSTING_STREAM,SRC_RECORD_ID,SRC_APP_CD,BUSINESS_DATE,AS_OF_DATE,REJECTION_TYPE,REJECTION_DETAIL


## Recon

`recon.reconcile(workflow_run_id)` reads `interface.trial_balance` and `gl.posting` for
the workflow — the GL side through the real `GLClient` abstraction, not a separate query
— aggregates each side to a balance per `RECON_KEYS` grain (`WORKFLOW_RUN_ID`,
`AS_OF_DATE`, and the nine GL segments plus `ACCOUNTED_CURRENCY`), and persists the
comparison to `recon.result`.

This runs under its own `RECON / RECONCILE` execution, created under the same
`workflow_run_id` being reconciled. That execution's own `run_id` is stamped as
`PRODUCER_RUN_ID` on every row it writes — distinct from `gl_run_id` above, which is GL's
own producer lineage for the postings being compared.

In [17]:
recon_result = recon.reconcile(workflow_run_id)

recon_run_id = recon_result.producer_run_id

print(
    f"result_count={recon_result.result_count} "
    f"break_count={recon_result.break_count} "
    f"producer_run_id={recon_run_id}"
)

result_count=7 break_count=0 producer_run_id=e45ec05e-8948-4e96-b853-4c4ee9cf2783


In [18]:
recon_df = recon.get_results(workflow_run_id)

display_df(recon_df)

,RECON_RESULT_ID,RECONCILED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,AS_OF_DATE,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,INTERFACE_BALANCE,GL_BALANCE,DIFFERENCE_AMOUNT
0,491bf38d-7ade-4175-a9b8-5540db7a491d,2026-08-28 10:37:09.602955,764781eb-c9e0-4aa7-9eba-9736a81a662b,e45ec05e-8948-4e96-b853-4c4ee9cf2783,2026-03-31,1000,1200,NYC,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-25000.000000000000,-25000.000000000000,0E-12
1,1d882e7c-424a-418f-92cd-d2b724270bb9,2026-08-28 10:37:09.602920,764781eb-c9e0-4aa7-9eba-9736a81a662b,e45ec05e-8948-4e96-b853-4c4ee9cf2783,2026-03-31,1000,1100,NYC,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-65000.000000000000,-65000.000000000000,0E-12
2,9828a307-b784-47fd-a185-07ff899ad2a1,2026-08-28 10:37:09.602932,764781eb-c9e0-4aa7-9eba-9736a81a662b,e45ec05e-8948-4e96-b853-4c4ee9cf2783,2026-03-31,1000,1200,NYC,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,USD,20000.000000000000,20000.000000000000,0E-12
3,067e276f-5f21-4d41-b12d-6616a6bf2546,2026-08-28 10:37:09.602896,764781eb-c9e0-4aa7-9eba-9736a81a662b,e45ec05e-8948-4e96-b853-4c4ee9cf2783,2026-03-31,1000,1100,NYC,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,90000.000000000000,90000.000000000000,0E-12
4,d6186e6e-eb07-4add-b12a-6468fa4c2292,2026-08-28 10:37:09.602943,764781eb-c9e0-4aa7-9eba-9736a81a662b,e45ec05e-8948-4e96-b853-4c4ee9cf2783,2026-03-31,1000,1200,NYC,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,USD,-20000.000000000000,-20000.000000000000,0E-12
5,966757f4-61d5-4ae2-af83-c11fc04ab745,2026-08-28 10:37:09.602977,764781eb-c9e0-4aa7-9eba-9736a81a662b,e45ec05e-8948-4e96-b853-4c4ee9cf2783,2026-03-31,2000,2100,TOR,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,USD,-38250.000000000000,-38250.000000000000,0E-12
6,401db9b4-29d5-49c4-8a47-1627bde2e86c,2026-08-28 10:37:09.602966,764781eb-c9e0-4aa7-9eba-9736a81a662b,e45ec05e-8948-4e96-b853-4c4ee9cf2783,2026-03-31,2000,2100,TOR,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,USD,38250.000000000000,38250.000000000000,0E-12


### Breaks

Rows with a non-zero `DIFFERENCE_AMOUNT` — expected to be empty for this balanced
synthetic run.

In [19]:
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

display_df(breaks_df)

,RECON_RESULT_ID,RECONCILED_AT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,AS_OF_DATE,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,AFFILIATE_CD,PRODUCT_CD,BOOK_CD,SOURCE_CD,ACCOUNTED_CURRENCY,INTERFACE_BALANCE,GL_BALANCE,DIFFERENCE_AMOUNT


In [20]:
spark.stop()